In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import random

In [ ]:
class Ocean:
    def __init__(self, size):
        self.size = size
        self.animals = set()
        self.animal_grid = {}

        # 0: empty, 1: plankton, 2: fish, 3: shark, 4: orca
        self.lattice = np.zeros((size, size), dtype=np.int8)
        self.plankton_lattice = np.zeros((size, size), dtype=np.int8)

    def add_animal(self, animal):
        if self.lattice[animal.x, animal.y] == 0:
            self.animals.add(animal)
            self.animal_grid[(animal.x, animal.y)] = animal
            self.lattice[animal.x, animal.y] = animal.get_type_id()
            return True
        return False

    def remove_animal(self, animal):
        if animal in self.animals:
            self.animals.remove(animal)
            if self.animal_grid.get((animal.x, animal.y)) is animal:
                self.animal_grid.pop((animal.x, animal.y), None)
                self.lattice[animal.x, animal.y] = 0

    def grow_plankton(self, repro_rate):
        empty_spots = (self.plankton_lattice == 0)
        plankton_mask = (self.plankton_lattice == 1)

        growth_attempts = (np.random.random((self.size, self.size)) < repro_rate)
        active_growth = plankton_mask & growth_attempts
        
        if not np.any(active_growth):
            return
        
        spawn_n = np.roll(active_growth, -1, axis=0)
        spawn_s = np.roll(active_growth, 1, axis=0)
        spawn_e = np.roll(active_growth, -1, axis=1)
        spawn_w = np.roll(active_growth, 1, axis=1)
        
        potential_spawns = spawn_n | spawn_s | spawn_e | spawn_w
        self.plankton_lattice[potential_spawns & empty_spots] = 1

    def find_target(self, x, y, target_type=0, layer='animal'):
        grid = self.lattice if layer == 'animal' else self.plankton_lattice
        dirs = [(0, 1), (0, -1), (1, 0), (-1, 0)]
        random.shuffle(dirs)

        for dx, dy in dirs:
            nx, ny = (x + dx) % self.size, (y + dy) % self.size
            if grid[nx, ny] == target_type:
                return (nx, ny)
        return None

class Animal:
    def __init__(self, x, y, repro_time, max_energy, energy=None):
        self.x, self.y = x, y
        self.timer = 0
        self.repro_limit = repro_time
        self.max_energy = max_energy

        if energy is None:
            self.energy = max_energy
        else:
            self.energy = energy

    def get_type_id(self): return 0

    def move(self, ocean, new_pos, is_hunting=False):
        old_pos = (self.x, self.y)

        if not is_hunting and ocean.lattice[new_pos] != 0:
            self.timer += 1
            return

        if ocean.animal_grid.get(old_pos) is self:
            ocean.animal_grid.pop(old_pos, None)
            ocean.lattice[old_pos] = 0

        self.x, self.y = new_pos
        ocean.animal_grid[new_pos] = self
        ocean.lattice[new_pos] = self.get_type_id()
        
        self.timer += 1
        if self.timer >= self.repro_limit:
            self.timer = 0
            baby = self.__class__(old_pos[0], old_pos[1], self.repro_limit, self.max_energy, (self.energy + self.max_energy) // 2)
            ocean.add_animal(baby)

    def hunt_step(self, ocean, prey_rewards):
        for prey_id, (repro_bonus, energy_bonus) in prey_rewards.items():
            is_plankton = (prey_id == 1)
            layer = 'plankton' if is_plankton else 'animal'
            target = ocean.find_target(self.x, self.y, target_type=prey_id, layer=layer)
        
            if target:
                if is_plankton:
                    ocean.plankton_lattice[target] = 0
                else:
                    victim = ocean.animal_grid.get(target)
                    if victim:
                        ocean.remove_animal(victim)
                    else:
                        continue
                      
                self.energy += energy_bonus
                if self.energy > self.max_energy:
                    self.timer += (self.energy - self.max_energy) // 2
                    self.energy = self.max_energy

                self.timer += repro_bonus
                self.move(ocean, target, is_hunting=True)                
                return

        self.energy -= 1
        if self.energy <= 0:
            ocean.remove_animal(self)
            return

        free_spot = ocean.find_target(self.x, self.y, target_type=0)
        if free_spot:
            self.move(ocean, free_spot)
        else:
            self.timer += 1

class Fish(Animal):
    def get_type_id(self): return 2
    
    def step(self, ocean):
        self.hunt_step(ocean, {1: (1, 4)})

class Shark(Animal):
    def get_type_id(self): return 3
    
    def step(self, ocean):
        self.hunt_step(ocean, {2: (6, 8)})

class Orca(Animal):
    def get_type_id(self): return 4
    
    def step(self, ocean):
        self.hunt_step(ocean, {3: (6, 12), 2: (0, 2)})

In [ ]:
frames = 500
fps = 10.0

size = 100
ocean = Ocean(size)
ocean.plankton_lattice = np.random.choice([0, 1], size=(size, size), p=[.96, .04]).astype(np.int8)

def populate(cls, count, r, e):
    for _ in range(count):
        while True:
            x, y = random.randint(0, size-1), random.randint(0, size-1)
            if ocean.lattice[x, y] == 0:
                ocean.add_animal(cls(x, y, r, e))
                break

plankton_repro_rate = 0.3
populate(Fish, 1400, 12, 4)
populate(Shark, 200, 24, 8)
populate(Orca, 100, 48, 16)


fig, ax = plt.subplots(figsize=(8, 8))
cmap = plt.get_cmap('ocean', 5)
combined = np.where(ocean.lattice > 0, ocean.lattice, ocean.plankton_lattice)
im = ax.imshow(combined, cmap=cmap, vmin=0, vmax=4, interpolation='nearest')
ax.axis('off')

history = []
def update(frame):
    ocean.grow_plankton(repro_rate=plankton_repro_rate)
    current_animals = list(ocean.animals)
    # random.shuffle(current_animals)

    for animal in current_animals:
        if animal in ocean.animals:
            animal.step(ocean)

    combined = np.where(ocean.lattice > 0, ocean.lattice, ocean.plankton_lattice)
    im.set_array(combined)

    plankton_pop = np.sum(ocean.plankton_lattice == 1)
    fish_pop  = np.sum(ocean.lattice == 2)
    shark_pop = np.sum(ocean.lattice == 3)
    orca_pop  = np.sum(ocean.lattice == 4)
    history.append([plankton_pop, fish_pop, shark_pop, orca_pop])
    
    ax.set_title(f"Step: {frame:<4} | Plankton: {plankton_pop:<4} | Fishes: {fish_pop:<4} | Sharks: {shark_pop:<4} | Orcas: {orca_pop:<4}")
    return [im]

anim = animation.FuncAnimation(
    fig,
    update,
    frames=frames,
    interval=1000.0 / fps,
    blit=True
)

anim.save(f"../results/gifs/13_wator_ocean.gif", writer='pillow', fps=fps)
plt.close(fig)

In [ ]:
history = np.array(history)
t = np.arange(len(history))

h_min = history.min(axis=0)
h_max = history.max(axis=0)

norm_history = (history - h_min) / (h_max - h_min + 1e-5)

fig, ax = plt.subplots(figsize=(10, 6))
labels = ['Plankton', 'Fish', 'Shark', 'Orca']
colors = ['green', 'blue', 'red', 'black']

for i in range(4):
    ax.plot(t, norm_history[:, i], label=labels[i], color=colors[i], alpha=0.6)

ax.set_xlabel('Steps')
ax.set_ylabel('Normalized Population')
ax.legend()
plt.title('Wa-Tor relative population over time')
plt.savefig('../results/plots/13_wator_population.png')
plt.show()
plt.close()

In [ ]:
# --- Podstawowe zadanie ---

A, B , E = 3, 20, 3

class Fish(Animal):
    def get_type_id(self): return 2
    
    def step(self, ocean):
        self.hunt_step(ocean, {1: (0, 0)})

class Shark(Animal):
    def get_type_id(self): return 3
    
    def step(self, ocean):
        self.hunt_step(ocean, {2: (0, E)})

frames = 500
fps = 10.0

size = 100
ocean = Ocean(size)

plankton_repro_rate = 0.0
populate(Fish, 4000, A, frames + 1)
populate(Shark, 1000, B, E)


fig, ax = plt.subplots(figsize=(8, 8))
cmap = plt.get_cmap('ocean', 5)
im = ax.imshow(ocean.lattice, cmap=cmap, vmin=0, vmax=4, interpolation='nearest')
ax.axis('off')

history_def = []
def update(frame):
    ocean.grow_plankton(repro_rate=plankton_repro_rate)
    current_animals = list(ocean.animals)
    # random.shuffle(current_animals)

    for animal in current_animals:
        if animal in ocean.animals:
            animal.step(ocean)

    im.set_array(ocean.lattice)

    fish_pop  = np.sum(ocean.lattice == 2)
    shark_pop = np.sum(ocean.lattice == 3)
    history_def.append([fish_pop, shark_pop])
    
    ax.set_title(f"Step: {frame:<4} | Fishes: {fish_pop:<4} | Sharks: {shark_pop:<4}")
    return [im]

anim = animation.FuncAnimation(
    fig,
    update,
    frames=frames,
    interval=1000.0 / fps,
    blit=True
)

anim.save(f"../results/gifs/13_wator_ocean_default.gif", writer='pillow', fps=fps)
plt.close(fig)

In [ ]:
history_def = np.array(history_def)
t = np.arange(len(history_def))

h_min = history_def.min(axis=0)
h_max = history_def.max(axis=0)

norm_history_def = (history_def - h_min) / (h_max - h_min + 1e-5)

fig, ax = plt.subplots(figsize=(10, 6))
labels = ['Fish', 'Shark']
colors = ['blue', 'red']

for i in range(2):
    ax.plot(t, norm_history_def[:, i], label=labels[i], color=colors[i], alpha=0.6)

ax.set_xlabel('Steps')
ax.set_ylabel('Normalized Population')
ax.legend()
plt.title('Wa-Tor relative population over time')
plt.savefig('../results/plots/13_wator_population_default.png')
plt.show()
plt.close()